In [7]:
import cv2
import numpy as np
import os
import pandas as pd

video_folder = "videos"
gt_folder = "pseudo_gt_masks"

def iou(pred, gt):
    pred = pred > 0
    gt = gt > 0
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return inter / union if union > 0 else 0

def precision(pred, gt):
    pred = pred > 0
    gt = gt > 0
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, np.logical_not(gt)).sum()
    return tp / (tp + fp) if (tp + fp) > 0 else 0

def recall(pred, gt):
    pred = pred > 0
    gt = gt > 0
    tp = np.logical_and(pred, gt).sum()
    fn = np.logical_and(np.logical_not(pred), gt).sum()
    return tp / (tp + fn) if (tp + fn) > 0 else 0

def get_pred_mask(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    _, th = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((9,9), np.uint8))
    return th

def extract_category(filename):
    return filename.split("_frame")[0]

gt_files = sorted([
    f for f in os.listdir(gt_folder)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

results = {}
skipped = 0

for file in gt_files:

    gt_path = os.path.join(gt_folder, file)
    gt = cv2.imread(gt_path, 0)

    if gt is None:
        print("❌ GT read failed:", file)
        skipped += 1
        continue

    gt = cv2.resize(gt, (640, 480))

    category = extract_category(file)

    frame = None

    for vid in os.listdir(video_folder):

        if not vid.lower().endswith(('.mp4', '.avi', '.mov')):
            continue

        cap = cv2.VideoCapture(os.path.join(video_folder, vid))

        frame_idx = int(file.split("_frame_")[1].split(".")[0])
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

        ret, fr = cap.read()
        cap.release()

        if ret:
            frame = fr
            break

    if frame is None:
        print("⚠ No matching frame found for:", file)
        skipped += 1
        continue

    frame = cv2.resize(frame, (640, 480))

    pred = get_pred_mask(frame)


    i = iou(pred, gt)
    p = precision(pred, gt)
    r = recall(pred, gt)

    if category not in results:
        results[category] = {
            "iou": [],
            "precision": [],
            "recall": []
        }

    results[category]["iou"].append(i)
    results[category]["precision"].append(p)
    results[category]["recall"].append(r)


table = []

if len(results) == 0:
    print("\n⚠ WARNING: No valid matches found. Check filenames or videos.")
else:
    for cat, vals in results.items():

        table.append({
            "Degradation Type": cat,
            "Tested Frames": len(vals["iou"]),
            "Best IoU": round(max(vals["iou"]), 4),
            "Average IoU": round(np.mean(vals["iou"]), 4),
            "Average Precision": round(np.mean(vals["precision"]), 4),
            "Average Recall": round(np.mean(vals["recall"]), 4)
        })

df = pd.DataFrame(table)

print("\n TABLE B – VIDEO WISE PERFORMANCE\n")
print(df)

df.to_csv("tableB_results.csv", index=False)

print("\n Saved: tableB_results.csv")
print("⚠ Skipped frames:", skipped)


 TABLE B – VIDEO WISE PERFORMANCE

      Degradation Type  Tested Frames  Best IoU  Average IoU  \
0    AlligatorCracking              6    0.2916       0.1093   
1        EdgeCracking3              1    0.0960       0.0960   
2        EdgeCracking4              6    0.0780       0.0653   
3        EdgeCracking9              4    0.0608       0.0467   
4        RoadPotholes2              1    0.0838       0.0838   
5        RoadPotholes3              3    0.1207       0.0935   
6         RoadPotholes              5    0.0929       0.0880   
7  pedestrianCrossing2              4    0.0881       0.0615   
8   pedestrianCrossing              3    0.1031       0.0741   

   Average Precision  Average Recall  
0             0.2147          0.1712  
1             0.1516          0.2074  
2             0.1060          0.1538  
3             0.0800          0.1076  
4             0.1390          0.1744  
5             0.2103          0.1442  
6             0.1227          0.2554  
7          